# Common Settings

In [1]:
import transformers
transformers.logging.set_verbosity_error()

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# LLM Tokenization - Downloading and Running An LLM

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="mps",  # Use MPS for Apple Silicon instead of "cuda"
    torch_dtype="auto",
    local_files_only=True,
    attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    local_files_only=True
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_ids=inputs['input_ids']

generation_output = model.generate(
    input_ids=input_ids, 
    attention_mask=inputs['attention_mask'], 
    max_new_tokens=20
)

In [4]:
print(input_ids[0])

tensor([14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
          293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
          372,  9559, 29889, 32001], device='mps:0')


In [5]:
for id in input_ids[0]:
    print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [6]:
print(tokenizer.decode(input_ids[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>


In [7]:
print(generation_output[0])

tensor([14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
          293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
          372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
         6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
           13,    13, 29928,   799], device='mps:0')


In [8]:
for id in generation_output[0]:
    print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>
Sub
ject
:
Heart
f
elt
Ap
ologies
for
the
Garden
ing
M
ish
ap






D
ear


In [9]:
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


In [10]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))

Sub
ject
Subject


# LLM Tokenization - Comparing Trained LLM Tokenizers

In [11]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    print(f"========== Loading: {tokenizer_name} ==========")
    try:
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    except Exception as e:
        print(f"Error loading tokenizer: {e}")
        return

    # --- 1. Metadata (Class & Algorithm) ---
    cls_name = type(tokenizer).__name__
    
    # Attempt to detect algorithm for Fast tokenizers
    if tokenizer.is_fast and hasattr(tokenizer, "backend_tokenizer"):
        algo_type = type(tokenizer.backend_tokenizer.model).__name__
    else:
        algo_type = "Standard (Python/Slow)"

    # --- 2. Gather Data ---
    all_special_ids = set(tokenizer.all_special_ids)
    added_vocab_map = tokenizer.get_added_vocab()
    
    # Filter 'added_vocab' to find tokens that are NOT in 'all_special_ids'
    # These are specific merges or custom tokens not mapped to roles like [CLS]
    additional_added_tokens = {
        k: v for k, v in added_vocab_map.items() 
        if v not in all_special_ids
    }

    # --- 3. Print Overview ---
    print(f"Class:          {cls_name}")
    print(f"Algorithm:      {algo_type}")
    print(f"Vocab Size:     {tokenizer.vocab_size}")
    
    print(f"\n[Token Statistics]")
    print(f"Total Special Tokens:   {len(all_special_ids)}")
    print(f"Total Added Tokens:     {len(added_vocab_map)}")
    print(f"  L Purely Additional:  {len(additional_added_tokens)} (Excluded from Special Roles)")

    # --- 4. Special Token Roles ---
    if tokenizer.special_tokens_map:
        print(f"\n[Special Token Roles]")
        print(f"{'Role':<25} | {'Token':<30} | {'ID'}")
        print("-" * 70)

        for role, value in tokenizer.special_tokens_map.items():
            # Handle list values (e.g. multiple additional_special_tokens)
            tokens_to_print = value if isinstance(value, list) else [value]
            
            for t in tokens_to_print:
                # Safe conversion in case a string in the map isn't in vocab
                t_id = tokenizer.convert_tokens_to_ids(t) if t in tokenizer.get_vocab() or t in added_vocab_map else "N/A"
                print(f"{role:<25} | {str(t):<30} | {t_id}")
    else:
        print("(No special tokens map found)")

    # --- 5. Additional Added Vocab (Filtered) ---
    if additional_added_tokens:
        print(f"\n[Additional Added Tokens] (Excluding Special Tokens)")
        
        # Sort by ID for cleaner viewing
        sorted_additional = sorted(additional_added_tokens.items(), key=lambda item: item[1])

        print(f"{'Token':<25} | {'ID'}")
        print("-" * 40)
        for token, t_id in sorted_additional:
            print(f"{token:<25} | ID: {t_id}")
    else:
        print(f"\n[Additional Added Tokens]")
        print("None. (All added tokens are currently classified as Special Tokens).")
    
    token_ids = tokenizer(sentence).input_ids
    print(f"\n[Token Usage]")
    print(f"Token Count:           {len(token_ids)}")
    print(f"Distinct Token Count:  {len(set(token_ids))}\n")
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [13]:
show_tokens(text, "bert-base-uncased")

========== Loading: bert-base-uncased ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31adb6bd0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 7cd6227e-96c3-497f-b360-a88b46ca4734)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31b0b8260>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: b2c37704-e995-445e-ac79-5c20abf22890)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 

Class:          BertTokenizerFast
Algorithm:      WordPiece
Vocab Size:     30522

[Token Statistics]
Total Special Tokens:   5
Total Added Tokens:     5
  L Purely Additional:  0 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
unk_token                 | [UNK]                          | 100
sep_token                 | [SEP]                          | 102
pad_token                 | [PAD]                          | 0
cls_token                 | [CLS]                          | 101
mask_token                | [MASK]                         | 103

[Additional Added Tokens]
None. (All added tokens are currently classified as Special Tokens).

[Token Usage]
Token Count:           41
Distinct Token Count:  29

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 *

In [14]:
show_tokens(text, "bert-base-cased")

========== Loading: bert-base-cased ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-cased/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31b0b9340>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 1329fbed-e4b8-49ec-84b5-55019eb46aad)')' thrown while requesting HEAD https://huggingface.co/bert-base-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-cased/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31b0b8680>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: b61a9f3b-e367-47df-9e00-15fede1a955a)')' thrown while requesting HEAD https://huggingface.co/bert-base-cased/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(

Class:          BertTokenizerFast
Algorithm:      WordPiece
Vocab Size:     28996

[Token Statistics]
Total Special Tokens:   5
Total Added Tokens:     5
  L Purely Additional:  0 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
unk_token                 | [UNK]                          | 100
sep_token                 | [SEP]                          | 102
pad_token                 | [PAD]                          | 0
cls_token                 | [CLS]                          | 101
mask_token                | [MASK]                         | 103

[Additional Added Tokens]
None. (All added tokens are currently classified as Special Tokens).

[Token Usage]
Token Count:           49
Distinct Token Count:  38

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Thre

In [15]:
show_tokens(text, "google/flan-t5-small")

========== Loading: google/flan-t5-small ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/flan-t5-small/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x3517d3230>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 47df78fa-53ee-430d-a2de-7c27c476edd2)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-small/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/flan-t5-small/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x3517d2d50>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: ad58633b-0406-4782-9e19-15dbf6e36d13)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-small/resolve/main/tokenizer_config.json
Retrying i

Class:          T5TokenizerFast
Algorithm:      Unigram
Vocab Size:     32100

[Token Statistics]
Total Special Tokens:   103
Total Added Tokens:     103
  L Purely Additional:  0 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
eos_token                 | </s>                           | 1
unk_token                 | <unk>                          | 2
pad_token                 | <pad>                          | 0
additional_special_tokens | <extra_id_0>                   | 32099
additional_special_tokens | <extra_id_1>                   | 32098
additional_special_tokens | <extra_id_2>                   | 32097
additional_special_tokens | <extra_id_3>                   | 32096
additional_special_tokens | <extra_id_4>                   | 32095
additional_special_tokens | <extra_id_5>                   | 32094
additional_special_tokens | <extra_id_6> 

In [16]:
show_tokens(text, "gpt2")

========== Loading: gpt2 ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /gpt2/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31b0b96d0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: bf1aefc2-13fe-498b-9f04-788996d5676b)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /gpt2/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31b0b8800>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: f2ffdc82-081c-4729-9ee7-218027d52166)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError("HTTPSConnectionPool(host='hug

Class:          GPT2TokenizerFast
Algorithm:      BPE
Vocab Size:     50257

[Token Statistics]
Total Special Tokens:   1
Total Added Tokens:     1
  L Purely Additional:  0 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
bos_token                 | <|endoftext|>                  | 50256
eos_token                 | <|endoftext|>                  | 50256
unk_token                 | <|endoftext|>                  | 50256

[Additional Added Tokens]
None. (All added tokens are currently classified as Special Tokens).

[Token Usage]
Token Count:           55
Distinct Token Count:  39


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

In [17]:
show_tokens(text, "Xenova/gpt-4")

========== Loading: Xenova/gpt-4 ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Xenova/gpt-4/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x351d27fb0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 6670f4fc-c29e-4e97-80f3-e25e9036687f)')' thrown while requesting HEAD https://huggingface.co/Xenova/gpt-4/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Xenova/gpt-4/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x351d70350>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 89104921-0879-440a-a927-9b5e1124c163)')' thrown while requesting HEAD https://huggingface.co/Xenova/gpt-4/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryErro

Class:          GPT2TokenizerFast
Algorithm:      BPE
Vocab Size:     100263

[Token Statistics]
Total Special Tokens:   1
Total Added Tokens:     7
  L Purely Additional:  6 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
bos_token                 | <|endoftext|>                  | 100257
eos_token                 | <|endoftext|>                  | 100257
unk_token                 | <|endoftext|>                  | 100257

[Additional Added Tokens] (Excluding Special Tokens)
Token                     | ID
----------------------------------------
<|fim_prefix|>            | ID: 100258
<|fim_middle|>            | ID: 100259
<|fim_suffix|>            | ID: 100260
<|im_start|>              | ID: 100264
<|im_end|>                | ID: 100265
<|endofprompt|>           | ID: 100276

[Token Usage]
Token Count:           41
Distinct Token Count:  35


 Eng

In [18]:
show_tokens(text, "bigcode/starcoder2-15b")

========== Loading: bigcode/starcoder2-15b ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bigcode/starcoder2-15b/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x35325f1d0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 681cf5a9-aa38-42fa-9bea-6399973dd314)')' thrown while requesting HEAD https://huggingface.co/bigcode/starcoder2-15b/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bigcode/starcoder2-15b/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x35325eea0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: e9ad2c63-9a82-4ce7-91c1-c9e65486c239)')' thrown while requesting HEAD https://huggingface.co/bigcode/starcoder2-15b/resolve/main/tokenizer_config.json
Re

Class:          GPT2TokenizerFast
Algorithm:      BPE
Vocab Size:     49152

[Token Statistics]
Total Special Tokens:   38
Total Added Tokens:     38
  L Purely Additional:  0 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
bos_token                 | <|endoftext|>                  | 0
eos_token                 | <|endoftext|>                  | 0
unk_token                 | <|endoftext|>                  | 0
additional_special_tokens | <|endoftext|>                  | 0
additional_special_tokens | <fim_prefix>                   | 1
additional_special_tokens | <fim_middle>                   | 2
additional_special_tokens | <fim_suffix>                   | 3
additional_special_tokens | <fim_pad>                      | 4
additional_special_tokens | <repo_name>                    | 5
additional_special_tokens | <file_sep>                     | 6
additi

In [19]:
show_tokens(text, "facebook/galactica-1.3b")

========== Loading: facebook/galactica-1.3b ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/galactica-1.3b/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x3521f2ff0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 8d80e5c9-d6e8-406a-8fc9-c5ac8bcab2b4)')' thrown while requesting HEAD https://huggingface.co/facebook/galactica-1.3b/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/galactica-1.3b/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x3521f2bd0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 6a6507c0-7337-4410-87ee-1aa652b5e35d)')' thrown while requesting HEAD https://huggingface.co/facebook/galactica-1.3b/resolve/main/tokenizer_config.jso

Class:          PreTrainedTokenizerFast
Algorithm:      BPE
Vocab Size:     50000

[Token Statistics]
Total Special Tokens:   0
Total Added Tokens:     23
  L Purely Additional:  23 (Excluded from Special Roles)
(No special tokens map found)

[Additional Added Tokens] (Excluding Special Tokens)
Token                     | ID
----------------------------------------
<s>                       | ID: 0
<pad>                     | ID: 1
</s>                      | ID: 2
<unk>                     | ID: 3
[START_REF]               | ID: 4
[END_REF]                 | ID: 5
[IMAGE]                   | ID: 6
<fragments>               | ID: 7
</fragments>              | ID: 8
<work>                    | ID: 9
</work>                   | ID: 10
[START_SUP]               | ID: 11
[END_SUP]                 | ID: 12
[START_SUB]               | ID: 13
[END_SUB]                 | ID: 14
[START_DNA]               | ID: 15
[END_DNA]                 | ID: 16
[START_AMINO]             | ID: 17
[END_AMINO] 

In [20]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

========== Loading: microsoft/Phi-3-mini-4k-instruct ==========


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /microsoft/Phi-3-mini-4k-instruct/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x351970f20>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 62e5eee8-786c-4e22-9925-b7ba9723f2a4)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /microsoft/Phi-3-mini-4k-instruct/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x351971010>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 2d85d16e-19be-42eb-ac8b-3ebfea20c49a)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instru

Class:          LlamaTokenizerFast
Algorithm:      BPE
Vocab Size:     32000

[Token Statistics]
Total Special Tokens:   3
Total Added Tokens:     14
  L Purely Additional:  11 (Excluded from Special Roles)

[Special Token Roles]
Role                      | Token                          | ID
----------------------------------------------------------------------
bos_token                 | <s>                            | 1
eos_token                 | <|endoftext|>                  | 32000
unk_token                 | <unk>                          | 0
pad_token                 | <|endoftext|>                  | 32000

[Additional Added Tokens] (Excluding Special Tokens)
Token                     | ID
----------------------------------------
</s>                      | ID: 2
<|assistant|>             | ID: 32001
<|placeholder1|>          | ID: 32002
<|placeholder2|>          | ID: 32003
<|placeholder3|>          | ID: 32004
<|placeholder4|>          | ID: 32005
<|system|>               

# Creating Contextualized Word Embeddings with Language Models

In [21]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /microsoft/deberta-base/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x31af005c0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 52982c51-3e13-43ad-963e-692177c63b7c)')' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /microsoft/deberta-base/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x351d27e30>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 026bcd98-82cc-4ce1-8812-94a92268aa01)')' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-base/resolve/main/tokenizer_config.json
Re

ConnectTimeout: HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /microsoft/deberta-v3-xsmall/resolve/main/model.safetensors (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x3522bc4d0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))

In [ ]:
output.shape

In [ ]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

In [ ]:
output

# Text Embeddings (For Sentences and Whole Documents)

In [ ]:
# pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

In [ ]:
vector.shape

# Word Embeddings Beyond LLMs

In [ ]:
# pip install gensim

In [ ]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

In [ ]:
model.most_similar([model['king']], topn=11)

# Training a Song Embedding Model

In [ ]:
# pip install pandas

In [ ]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [ ]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

In [ ]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [ ]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

In [ ]:
print(songs_df.iloc[2172])

In [ ]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=10)
    )[:,0]
    return songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)